# Predicting credit card churn

Which customers are likely to leave the bank?

Steps:
1. Load the cleaned data and explore it
2. Split into train and test **first**, so the test set is never used for any decision
3. Pick features with an Extra Trees model (on the training data only)
4. Compare 5 models. Scaling and SMOTE run **inside** a pipeline, so cross-validation only balances the training folds
5. Evaluate the best model on the test set

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

RANDOM_STATE = 42

## 1. Load and explore

In [ ]:
data = pd.read_csv('../cleaned_data/cleaned_data.csv')
print(data.shape)
print(f"Churn rate: {data['Attrition_Flag'].mean():.1%}")

In [ ]:
numeric = data.select_dtypes(include='number')

plt.figure(figsize=(15, 10))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', center=0, cmap='viridis', annot_kws={'size': 9})
plt.title('Feature correlation', fontsize=16)
plt.show()

plt.figure(figsize=(18, 18))
for i, col in enumerate(numeric.columns):
    ax = plt.subplot(4, 4, i + 1)
    sns.histplot(data=data, x=col, kde=True, ax=ax)
    ax.set_title(col, fontsize=12)
plt.tight_layout()
plt.show()

## 2. Encode and split (before anything is learned from the data)

In [ ]:
categorical_cols = ['Education_Level', 'Income_Category', 'Marital_Status', 'Card_Category']
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

X = data.drop(columns='Attrition_Flag')
y = data['Attrition_Flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {len(X_train)} customers, test: {len(X_test)} customers ({y_test.sum()} churned)")

## 3. Feature selection on the training data only

In [ ]:
selector = ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_STATE)
selector.fit(X_train, y_train)

importances = pd.Series(selector.feature_importances_, index=X_train.columns).sort_values(ascending=False)
selected_features = importances[importances > importances.mean()].index

X_train = X_train[selected_features]
X_test = X_test[selected_features]
print(f"{len(selected_features)} features kept:", list(selected_features))

## 4. Compare models

Each model is a pipeline: scaler -> SMOTE -> model.
SMOTE only runs when the pipeline is fitted, so during cross-validation the validation fold keeps its real class balance.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(criterion='entropy', random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', model),
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['recall', 'f1'])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    pipelines[name] = pipe

    rows.append({
        'Model': name,
        'CV churn recall': scores['test_recall'].mean(),
        'CV churn F1': scores['test_f1'].mean(),
        'Test accuracy': accuracy_score(y_test, y_pred),
        'Test churn precision': precision_score(y_test, y_pred),
        'Test churn recall': recall_score(y_test, y_pred),
        'Test churn F1': f1_score(y_test, y_pred),
    })

results = pd.DataFrame(rows).set_index('Model').sort_values('CV churn F1', ascending=False).round(2)
results

## 5. Best model on the test set

The best model is picked on the **cross-validation** churn F1 (training data), not on the test set.
F1 balances precision and recall for the churn class; accuracy is misleading because only ~16% of customers churn.

In [ ]:
best_name = results.index[0]
best_pipe = pipelines[best_name]
y_pred = best_pipe.predict(X_test)

print(f"Best model: {best_name}")
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Churned'], yticklabels=['Stayed', 'Churned'])
plt.title(f'Confusion matrix: {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print(f"Found {cm[1, 1]} of the {cm[1].sum()} customers who churned.")

## 6. Which features matter most

In [ ]:
top = importances[selected_features]

plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color='steelblue')
plt.title('Feature importance (Extra Trees, training data)')
plt.xlabel('Importance')
plt.ylabel('')
plt.show()